In [ ]:
from typing import Dict, Any
import torch
from torch import Generator, logspace
from tqdm import tqdm

from occhio.autoencoder import *
from occhio.distributions import *
from occhio.toy_model import ToyModel
from occhio.model_grid import ModelGrid, Axis
from occhio.visualization.phase_change import plot_phase_change

In [ ]:
N_FEATURES = 5
N_HIDDEN = 2
EXPERIMENT_SIZE = 32

In [ ]:
def create_model(
    params: Dict[str, Any] = {}, default_model: ToyModel | None = None, *args, **kwargs
) -> ToyModel:
    density = params["Density"]
    relative_importance = params["Importance"]
    # random_seed = params["Random Seeds"]

    generator = Generator(device="cpu").manual_seed(42)

    model = ToyModel(
        distribution=SparseUniform(
            N_FEATURES, p_active=density, device="cpu", generator=generator
        ),
        ae=TiedLinearRelu(
            N_FEATURES,
            N_HIDDEN,
            generator=generator,
            device="cpu",
        ),
        importances=relative_importance ** torch.arange(N_FEATURES),
        device="cpu",
    )
    return model

In [ ]:
densities = logspace(0, -2, EXPERIMENT_SIZE)
importances = logspace(-1, 1, EXPERIMENT_SIZE)
# random_seeds = torch.arange(10, 20, dtype=torch.float32)

model_grid = ModelGrid(
    create_model,
    axes=[
        Axis(label="Density", values=densities),
        Axis(label="Importance", values=importances),
        # Axis(label="Random Seeds", values=random_seeds),
    ],
)

In [ ]:
model_grid.fit(n_epochs=256)

In [ ]:
model_grid_2 = ModelGrid(
    create_model,
    axes=[
        Axis(label="Density", values=densities),
        Axis(label="Importance", values=importances),
        # Axis(label="Random Seeds", values=random_seeds),
    ],
)

In [ ]:
for model in model_grid_2.models.ravel():
    model.fit(n_epochs=100)